# Blogify-Smart Blogging Platform — Category classifier training

Trained on **your MongoDB blogs** (title + subtitle + description → category).

- **Features:** TF-IDF vectorization
- **Model:** Logistic Regression (scikit-learn)
- **Labels:** `category` field (includes **All**, Technology, Startup, etc.)

Run all cells. Metrics and classification report appear below. Artifacts are saved to `models/*.pkl` for the API.

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name != "ml-service":
    ROOT = ROOT / "ml-service"
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from train_model import fetch_labeled_blogs, run_training_pipeline

print("Working directory:", ROOT)

Working directory: c:\Users\pramo\OneDrive\Desktop\Blogify-ML\ml-service


## 1. Load dataset from MongoDB

In [ ]:
texts, labels = fetch_labeled_blogs()
print(f"Total samples: {len(texts)}")
print(f"Categories: {sorted(set(labels))}")

# Preview first row
if texts:
    print("\nExample text (first 200 chars):")
    print(texts[0][:200], "...")
    print("Label:", labels[0])

## 2. Train, evaluate, and save model

In [ ]:
metrics = run_training_pipeline(verbose=True)
metrics

## 3. Test a prediction

In [ ]:
from predict import predict_category

sample = predict_category(
    title="How AI is changing web development in 2026",
    content="<p>Machine learning, Python, and automation in software jobs.</p>",
)
sample

## Saved files

| File | Purpose |
|------|--------|
| `models/vectorizer.pkl` | Fitted TF-IDF |
| `models/model.pkl` | Fitted Logistic Regression |
| `models/label_encoder.pkl` | Category name ↔ class index |
| `models/training_metrics.json` | Accuracy, F1, report |
| `models/figures/*.png` | Report charts (run section 4) |

Start API: `uvicorn main:app --port 8000`

## 4. Report charts (accuracy, F1, confusion matrix, recommendations)

Run after training. Uses `models/training_metrics.json` and saved `.pkl` files.

**Kernel:** Select your `ml-service` `.venv` Python kernel (not system Python).

In [ ]:
%matplotlib inline

from generate_report_charts import generate_all_charts

# Shows all charts inline + saves PNGs to models/figures/
saved = generate_all_charts(show=True, save=True)
saved

### Individual charts (optional)

Run any cell below if you want one chart at a time.

In [ ]:
from generate_report_charts import (
    load_training_metrics,
    reproduce_test_predictions,
    plot_overall_metrics,
    plot_per_category_metrics,
    plot_confusion_matrix_chart,
    plot_dataset_distribution,
    plot_top_features,
    plot_recommender_similarity,
)
from train_model import fetch_labeled_blogs

metrics = load_training_metrics()
y_true, y_pred, class_names = reproduce_test_predictions()
_, labels = fetch_labeled_blogs()

plot_overall_metrics(metrics)
plot_per_category_metrics(metrics)
plot_confusion_matrix_chart(y_true, y_pred, class_names)
plot_dataset_distribution(labels)
plot_top_features()
plot_recommender_similarity()